# 11 · Swiss roll: AE + oráculo y cuadrícula perfectamente separada

Dos experimentos nuevos, con el mismo protocolo del 08 v2 (mismas réplicas, mismo β, mismas métricas), para que sus filas se puedan añadir a la tabla normalizada del artículo.

| | Pregunta | Diseño | Qué se reporta |
|---|---|---|---|
| **O** | **AE + oráculo.** ¿Qué hace un decodificador aprendido con la carta del oráculo, y la reconstrucción **conserva** la carta si se empieza exactamente en ella? | Encoder afín = lectura del oráculo ψ₀…ψ_{m−1} → (s, h), en unidades de la carta divididas por **una sola** escala isótropa (sin deformar los ejes). `oracle_ae`: encoder **fijo**, solo se entrena el decodificador. `oracle_ae_free`: mismo inicio, encoder **entrenable**, sin y con jac λ = 0.1. 12 réplicas (las mismas del AE en el 08 v2) | ‖x − x̂‖² frente a m, junto a tres referencias: oráculo afín → roll exacto, decodificador entrenado sobre la carta **verdadera** y ese mismo decodificador aplicado a los puntos del oráculo. Latente frente al oráculo (s y h normalizados): 0 para el fijo por construcción; para el libre mide **cuánto se aleja** la reconstrucción de la carta |
| **Q** | **Cuadrícula perfectamente separada.** ¿Cambian las conclusiones si los puntos están a la **misma distancia** en s y en h? | Rejilla 132 × 22 con paso Δ = 0.4545 en ambos ejes (N = 2904, celdas cuadradas, nodos a Δ/2 de los bordes); la cuadrícula del 08 v2 (136 × 22) tenía pasos 0.441 y 0.455. β con la misma regla sin etiquetas, recalculado para esta geometría. DMAP + AE continuación (λ = 0 y 0.1), AE sin DMAP (λ = 0 y 0.1) y AE + oráculo (fijo y libre). 6 réplicas (cambia solo la partición) | las mismas curvas, tabla y anillos |

**Por qué el encoder fijo no es trivial.** El latente es la carta del oráculo, así que el error frente al oráculo es 0. Lo que se aprende es el **decodificador**: su error ambiental, comparado con el del oráculo seguido del roll exacto, mide cuánto cuesta (o cuánto ayuda) un decodificador MLP sobre la carta afín. Es el término de comparación directo para DMAP + AE + jac, cuyo latente coincide con el oráculo.

**Por qué el encoder libre.** Empieza con el latente exactamente sobre la carta. Si la reconstrucción por sí sola seleccionara la carta, el encoder no tendría razón para moverse. Si se mueve, la reconstrucción prefiere otra parametrización.

Duración estimada con 2 × T4: ~3 h para O y ~2.6 h para Q, más ~15 min del barrido de β de la cuadrícula (cabe en una sesión de 12 h; usa **Save & Run All (Commit)**). Si adjuntas como *Input* la salida de una corrida anterior de este notebook, las configuraciones completas no se repiten.

## Preparación

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
ROOT = Path('/kaggle/working/dmap_vae_kaggle')
INPUT = Path('/kaggle/input')
CU = 'flash_diffusion.cu'
CODE = ['spectral.py', 'vae.py', 'data.py', 'experiment.py', 'launch.py', 'plots.py',
        'checks.py', 'prepare_alanine.py', 'generate_angles.py', 'SOURCE.json',
        'geometry.py', 'beta_sweep.py', 'protocol.py', 'config_beta_sweep.json',
        'config_beta_sweep_torus.json', 'config_beta_sweep_mnist.json', 'config_beta_sweep_alanine.json', 'latent_figure.py', 'export_figs.py']

# 1) Refresh the code on EVERY run (a partial copy left in /kaggle/working is never trusted).
#    The results in /kaggle/working/results_* are not touched.
sources = sorted({q.parent for q in INPUT.rglob('launch.py') if (q.parent / 'spectral.py').exists()})
archives = sorted(INPUT.rglob('dmap_vae_kaggle*.zip'))
if len(sources) > 1:
    raise FileNotFoundError(f'Hay varias copias del paquete en /kaggle/input: {sources}. Deja solo una.')
ROOT.mkdir(parents=True, exist_ok=True)
if sources:
    shutil.copytree(sources[0], ROOT, dirs_exist_ok=True)
elif len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(ROOT.parent)
else:
    raise FileNotFoundError('Añade el paquete dmap_vae_kaggle como Kaggle Dataset; debe haber una sola copia.')
missing_code = [f for f in CODE if not (ROOT / f).exists()]
assert not missing_code, (f'Faltan archivos del paquete: {missing_code}. Este notebook necesita el '
                          'paquete v5 (dmap_vae_kaggle_v5.zip): sustituye el Dataset anterior.')

# 2) The CUDA source. Kaggle drops or flattens subfolders when a Dataset is built from loose
#    files, so look for it in vendor/, flattened at the root, or inside any attached ZIP.
target = ROOT / 'vendor' / CU
target.parent.mkdir(exist_ok=True)
if not target.exists():
    found = sorted(INPUT.rglob(CU))
    if found:
        shutil.copy2(found[0], target)
    else:
        for arc in sorted(INPUT.rglob('*.zip')):
            with zipfile.ZipFile(arc) as z:
                names = [n for n in z.namelist() if n.endswith('/' + CU) or n == CU]
                if names:
                    target.write_bytes(z.read(names[0]))
                    break
    lic = sorted(INPUT.rglob('LICENSE.FlashDiffusion'))
    if lic and not (target.parent / lic[0].name).exists():
        shutil.copy2(lic[0], target.parent / lic[0].name)
if not target.exists():
    raise FileNotFoundError(
        f'No se encontró {CU} en /kaggle/input. Sube la carpeta vendor/ (o el ZIP completo) '
        'al Kaggle Dataset; al subir archivos sueltos Kaggle omite las subcarpetas.')
expected = json.loads((ROOT / 'SOURCE.json').read_text())['sha256']
actual = hashlib.sha256(target.read_bytes()).hexdigest()
assert actual == expected, f'{CU} no coincide con SOURCE.json ({actual} != {expected}).'
print('Fuente CUDA verificada:', target)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import importlib.util
missing = [m for m in ['numpy','scipy','pandas','matplotlib','ninja']
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch
assert torch.cuda.device_count() >= 2, 'Selecciona GPU T4 x2 en Kaggle.'
print([(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i)) for i in range(2)])
(ROOT / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))


## Controles del paquete (una GPU, ~3 min)

Compila FlashDiffusion y ejecuta `checks.py`: kernel contra matriz densa, convención de signo, métrica kNN, continuación con ceros, paso AE→VAE y los controles nuevos de v5 (regla de degeneración del toro, regla argmin, sonda de etiquetas de MNIST, rotación de trayectorias de alanina). Si falla, no sigas.

In [ ]:
import time
t0 = time.time()
r = subprocess.run([sys.executable, str(ROOT / 'checks.py'), '--backend', 'flash', '--device', 'cuda:0',
                    '--out', str(ROOT / 'checks_11.json')],
                   env=dict(os.environ, CUDA_VISIBLE_DEVICES='0'), capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:]); print(r.stderr[-6000:])
    raise RuntimeError('No continuar: falló un control del paquete.')
checks = json.loads((ROOT / 'checks_11.json').read_text())
print({k: v for k, v in checks.items() if isinstance(v, str)})
print(f"signo: {checks['sign_overlap_min']:.6f} | kNN suave/plegada: {checks['knn_smooth_ratio']:.2f} / "
      f"{checks['knn_folded_ratio']:.0f} veces el piso | {(time.time()-t0)/60:.1f} min")

## β

In [ ]:
import protocol as P
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
# beta del Swiss roll aleatorio: el MISMO factor del 08 v2, para que las réplicas sean comparables.
FACTOR = 64
for prev in [Path('/kaggle/working/beta_sweep_swiss/beta_selection.json')] + sorted(Path('/kaggle/input').rglob('beta_sweep_swiss/beta_selection.json')):
    if prev.exists():
        FACTOR = json.loads(prev.read_text())['selected_factor'] or FACTOR
        break
# beta de la cuadrícula perfectamente separada: misma regla sin etiquetas, su propia geometría.
cfg_q = json.loads((ROOT / 'config_beta_sweep.json').read_text())
cfg_q.update(grid='square', data_seeds=[7, 1001], modes=[2, 4, 8, 16, 64], out='/kaggle/working/beta_sweep_swiss_square')
found = sorted(Path('/kaggle/input').rglob('beta_sweep_swiss_square/beta_selection.json'))
sel_q = json.loads(found[0].read_text()) if found else P.run_sweep(cfg_q, ROOT, 'beta_sweep_swiss_square')
FACTOR_SQ = sel_q['selected_factor'] or FACTOR
if sel_q.get('edge_warning'):
    print('AVISO:', sel_q['edge_warning'])
print(f"factor beta: aleatorio = {FACTOR} | cuadrícula separada = {FACTOR_SQ} (admisibles {sel_q.get('admissible')})")

## Plan y estimación de tiempo

El orden es la prioridad: primero O (AE + oráculo) y después Q (cuadrícula). La referencia «decodificador sobre la carta verdadera» se entrena una vez por réplica, solo en `ORACLE_l0` y `SQ_ORACLE_l0`.

In [ ]:
import math
BASE = json.loads((ROOT / 'config_swiss.json').read_text())
BASE.update(width=60., height=10.)
COMMON = dict(P.COMMON)
LADDER = [2, 4, 8, 16, 64]
M = max(LADDER)
SEEDS_AE = list(range(12))      # las mismas réplicas del AE en el 08 v2 (data_seed = 1000 + semilla)
SEEDS_SQ = list(range(6))
comun = dict(beta_factor=FACTOR, modes=LADDER, m_protocols=['scratch'], mode_order='spectral',
             oracle_decoder=False, reg_k=10)
def entrada(tag, models, seeds, lam=0., **extra):
    p = dict(comun, tag=tag, models=models, seeds=seeds, out=f'/kaggle/working/results_swiss_v3_{tag}')
    p.update(extra)
    if lam > 0:
        p.update(reg='jac', reg_weight=lam)
    return p
PLAN = [
    # O: AE + oráculo (Swiss roll aleatorio)
    entrada('ORACLE_l0', ['oracle_ae', 'oracle_ae_free'], SEEDS_AE, oracle_decoder=True),
    entrada('ORACLE_l0.1', ['oracle_ae_free'], SEEDS_AE, 0.1),
    # Q: cuadrícula perfectamente separada
    entrada('SQ_AE_l0', ['ae'], SEEDS_SQ, grid='square', beta_factor=FACTOR_SQ, m_protocols=['scratch', 'warm'], scratch_first_only=True),
    entrada('SQ_AE_l0.1', ['ae'], SEEDS_SQ, 0.1, grid='square', beta_factor=FACTOR_SQ, m_protocols=['scratch', 'warm'], scratch_first_only=True),
    entrada('SQ_PLAIN_AE_l0', ['plainae'], SEEDS_SQ, grid='square', beta_factor=FACTOR_SQ),
    entrada('SQ_PLAIN_AE_l0.1', ['plainae'], SEEDS_SQ, 0.1, grid='square', beta_factor=FACTOR_SQ),
    entrada('SQ_ORACLE_l0', ['oracle_ae', 'oracle_ae_free'], SEEDS_SQ, grid='square', beta_factor=FACTOR_SQ, oracle_decoder=True),
    entrada('SQ_ORACLE_l0.1', ['oracle_ae_free'], SEEDS_SQ, 0.1, grid='square', beta_factor=FACTOR_SQ),
]
P.reuse_inputs(PLAN)
OUT = {p['tag']: Path(p['out']) for p in PLAN}
SECONDS_PER_MODEL = 80 * 1.4
pendientes = [p for p in PLAN if not (Path(p['out']) / 'metrics.csv').exists()]
P.show_plan(pendientes, 'swiss', SECONDS_PER_MODEL, COMMON) if pendientes else print('todo reutilizado')

## Ejecutar

Se puede re-ejecutar: las configuraciones completas se omiten y las parciales se señalan (bórralas o cambia el `tag`). Para una corrida larga conviene **Save Version → Save & Run All (Commit)**.

In [ ]:
import protocol as P
hechas, problemas = P.run_plan(pendientes, BASE, ROOT, common=COMMON)

## Tabla normalizada (mismo formato que la del 08 v2)

Columnas idénticas a `final_v2_tabla_normalizada.csv`, así que las filas nuevas se pueden **pegar debajo** de esa tabla:
- ejes s y h normalizados por su varianza (0 = perfecto, 1 = sin información);
- alineación por semejanza (E(2) + escala);
- IC 95 % por bootstrap de la mediana sobre réplicas.

Para el encoder fijo, «lat↔oráculo» debe ser ~0 (es un control de que el encoder es exactamente el oráculo).

In [ ]:
from latent_figure import latent_vs_oracle_table, rings_figure
rng_b = np.random.default_rng(0)
def mediana_ic(x, n=5000):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    if not len(x):
        return np.nan, np.nan, np.nan
    b = np.median(rng_b.choice(x, (n, len(x))), axis=1)
    return np.median(x), *np.quantile(b, [.025, .975])

ETIQ = {('ORACLE_l0', 'oracle_ae'): 'DMAP + oráculo (encoder fijo)',
        ('ORACLE_l0', 'oracle_ae_free'): 'DMAP + AE iniciado en el oráculo',
        ('ORACLE_l0.1', 'oracle_ae_free'): 'DMAP + AE iniciado en el oráculo + jac 0.1',
        ('SQ_AE_l0', 'ae_warm'): 'cuadrícula separada · DMAP + AE cont.',
        ('SQ_AE_l0.1', 'ae_warm'): 'cuadrícula separada · DMAP + AE cont. + jac 0.1',
        ('SQ_PLAIN_AE_l0', 'plainae'): 'cuadrícula separada · AE sin DMAP',
        ('SQ_PLAIN_AE_l0.1', 'plainae'): 'cuadrícula separada · AE sin DMAP + jac 0.1',
        ('SQ_ORACLE_l0', 'oracle_ae'): 'cuadrícula separada · DMAP + oráculo (encoder fijo)',
        ('SQ_ORACLE_l0', 'oracle_ae_free'): 'cuadrícula separada · DMAP + AE iniciado en el oráculo',
        ('SQ_ORACLE_l0.1', 'oracle_ae_free'): 'cuadrícula separada · DMAP + AE iniciado en el oráculo + jac 0.1'}
orden = list(ETIQ.values())
LT, AMB = [], []
for (tag, model), lab in ETIQ.items():
    d = P.load(OUT[tag])
    if d is None:
        print('falta', tag); continue
    t = latent_vs_oracle_table(OUT[tag], [model])
    if len(t):
        LT.append(t.assign(corrida=lab, tag=tag, modelo=model))
    q = P.with_chain_start(d)
    q = q[q.model == model]
    AMB.append(q[['seed', 'm', 'mse_x', 'unfolded', 'knn_chart_ratio', 'local_distortion']].assign(corrida=lab, tag=tag, modelo=model))
LT, AMB = pd.concat(LT), pd.concat(AMB)

COLS = [('latent_vs_oracle_s_norm', 'lat↔oráculo s'), ('latent_vs_oracle_h_norm', 'lat↔oráculo h'),
        ('latent_vs_truth_s_norm', 'lat↔verdad s'), ('latent_vs_truth_h_norm', 'lat↔verdad h'),
        ('oracle_vs_truth_s_norm', 'oráculo↔verdad s'), ('oracle_vs_truth_h_norm', 'oráculo↔verdad h'),
        ('latent_vs_oracle', 'lat↔oráculo (unid. carta)'), ('latent_vs_oracle_affine', 'lat↔oráculo afín')]
filas = []
for (lab, m), g in LT.groupby(['corrida', 'm']):
    fila = dict(corrida=lab, m=int(m), n=g.seed.nunique())
    for c, nombre in COLS:
        med, lo, hi = mediana_ic(g[c])
        fila[nombre] = med; fila[nombre + ' IC'] = f'{lo:.3g}–{hi:.3g}' if np.isfinite(lo) else ''
    a = AMB[AMB.corrida == lab]
    a = a[a.m == (a.m.min() if 'sin DMAP' in lab else m)]
    med, lo, hi = mediana_ic(a.mse_x)
    fila['‖x−x̂‖²'] = med; fila['‖x−x̂‖² IC'] = f'{lo:.3g}–{hi:.3g}'
    filas.append(fila)
TN = pd.DataFrame(filas)
TN = TN.iloc[sorted(range(len(TN)), key=lambda i: (orden.index(TN.corrida.iloc[i]), TN.m.iloc[i]))].reset_index(drop=True)
TN.to_csv('/kaggle/working/final_v3_tabla_normalizada.csv', index=False)
num = [n for _, n in COLS] + ['‖x−x̂‖²']
display(TN[TN.m == M][['corrida', 'n'] + [c for n in num for c in (n, n + ' IC')]]
        .style.format({n: '{:.3g}' for n in num}, na_rep='—')
        .set_caption(f'm = {M}: medianas e IC 95 % (bootstrap sobre réplicas)'))

# control: el encoder fijo ES el oráculo (error latente ~ 0 en todas las réplicas y m)
fijo = LT[LT.modelo == 'oracle_ae']
print('encoder fijo: máx. lat↔oráculo (unid. carta) =', float(fijo.latent_vs_oracle.max()))

## Referencias del error ambiental

Para cada geometría y cada m, mediana e IC 95 % de:
- **oráculo afín → roll exacto**: la carta del oráculo proyectada al rectángulo y enrollada con la fórmula exacta (sin red);
- **decodificador sobre la carta verdadera** (`chart_decoder`): el mismo MLP entrenado con (s, h) verdaderos; no depende de m;
- **ese decodificador aplicado a los puntos del oráculo** (`oracle_decoder`).

El encoder fijo se compara con las tres. La diferencia entre «encoder fijo» y «decodificador verdadero aplicado al oráculo» es lo que gana un decodificador que se adapta a los errores del oráculo.

In [ ]:
filas = []
for geo, tag in (('aleatorio', 'ORACLE_l0'), ('cuadrícula separada', 'SQ_ORACLE_l0')):
    d = P.load(OUT[tag])
    if d is None:
        continue
    cd = d[d.model == 'chart_decoder'].mse_x
    for m in LADDER:
        fila = dict(geometria=geo, m=m)
        for nombre, serie in (('oráculo → roll exacto', d[(d.model == 'oracle') & (d.m == m)].oracle_exact_mse_x),
                              ('decoder carta verdadera', cd),
                              ('decoder verdadero sobre oráculo', d[(d.model == 'oracle_decoder') & (d.m == m)].mse_x),
                              ('DMAP + oráculo (encoder fijo)', AMB[(AMB.tag == tag) & (AMB.modelo == 'oracle_ae') & (AMB.m == m)].mse_x),
                              ('AE iniciado en el oráculo', AMB[(AMB.tag == tag) & (AMB.modelo == 'oracle_ae_free') & (AMB.m == m)].mse_x)):
            med, lo, hi = mediana_ic(serie)
            fila[nombre] = med; fila[nombre + ' IC'] = f'{lo:.3g}–{hi:.3g}' if np.isfinite(lo) else ''
        filas.append(fila)
REFS = pd.DataFrame(filas)
REFS.to_csv('/kaggle/working/final_v3_referencias_ambiente.csv', index=False)
display(REFS.style.format({c: '{:.4g}' for c in REFS.columns if not c.endswith('IC') and c not in ('geometria', 'm')})
        .set_caption('Error ambiental ‖x − x̂‖² (test): medianas e IC 95 %'))

In [ ]:
# ---- vistazo rápido: ambiente y latente frente a m ----
estilo = {'oracle_ae': ('C0', 's', '-'), 'oracle_ae_free': ('C3', 'o', '-'), 'ae_warm': ('C7', 'o', '-'), 'plainae': ('C1', None, '--')}
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for row, (geo, pref) in enumerate((('aleatorio', 'ORACLE'), ('cuadrícula separada', 'SQ_'))):
    a, b, c = axes[row]
    for (tag, model), lab in ETIQ.items():
        if not tag.startswith(pref) or tag not in set(AMB.tag):
            continue
        col, mk, ls = estilo[model]
        if tag.endswith('0.1'):
            ls = ':'
        g = AMB[(AMB.tag == tag) & (AMB.modelo == model)].groupby('m').mse_x.median()
        if model == 'plainae':
            a.axhline(g.iloc[0], color=col, ls=ls, label=lab)
        else:
            a.plot(g.index, g.values, color=col, marker=mk, ls=ls, label=lab)
        for ax, cc in ((b, 'latent_vs_oracle_s_norm'), (c, 'latent_vs_oracle_h_norm')):
            l = LT[(LT.tag == tag) & (LT.modelo == model)].groupby('m')[cc].median()
            ax.plot(l.index, np.maximum(l.values, 1e-12), color=col, marker=mk or '.', ls=ls, label=lab)
    r = REFS[REFS.geometria == geo]
    if len(r):
        a.plot(r.m, r['oráculo → roll exacto'], 'k--', lw=1, label='oráculo → roll exacto')
        a.plot(r.m, r['decoder verdadero sobre oráculo'], 'k-.', lw=1, label='decoder verdadero sobre oráculo')
        a.axhline(r['decoder carta verdadera'].iloc[0], color='k', ls=':', lw=1, label='decoder carta verdadera')
    a.set(title=f'{geo}: ‖x − x̂‖²', ylabel='MSE por coordenada (test)')
    b.set(title=f'{geo}: latente vs oráculo, s', ylabel='error / Var(s)')
    c.set(title=f'{geo}: latente vs oráculo, h', ylabel='error / Var(h)')
    for ax in (a, b, c):
        ax.set(xscale='log', yscale='log', xlabel='m (modos DMAP)')
        ax.set_xticks(LADDER); ax.set_xticklabels([str(m) for m in LADDER]); ax.minorticks_off()
    a.legend(fontsize=6); b.legend(fontsize=6)
fig.tight_layout()
fig.savefig('/kaggle/working/final_v3_curvas.png', dpi=200); fig.savefig('/kaggle/working/final_v3_curvas.pdf')
plt.show()

# ---- anillos (vistazo) ----
for nombre, filas_r in (('oraculo', [(OUT['ORACLE_l0'], 'oracle_ae', 'encoder fijo'), (OUT['ORACLE_l0'], 'oracle_ae_free', 'iniciado en oráculo'),
                                    (OUT['ORACLE_l0.1'], 'oracle_ae_free', 'iniciado en oráculo + jac 0.1')]),
                        ('cuadricula_separada', [(OUT['SQ_AE_l0'], 'ae_warm', 'DMAP + AE cont.'), (OUT['SQ_AE_l0.1'], 'ae_warm', 'DMAP + AE cont. + jac 0.1'),
                                                 (OUT['SQ_PLAIN_AE_l0'], 'plainae', 'AE sin DMAP'), (OUT['SQ_ORACLE_l0'], 'oracle_ae_free', 'iniciado en oráculo')])):
    rows = [dict(label=lab, folder=f, model=mo) for f, mo, lab in filas_r if P.load(f) is not None]
    if rows:
        rings_figure(rows, [4, 8, 16, 64], f'/kaggle/working/final_v3_anillos_{nombre}',
                     f'{nombre}: puntos del oráculo (círculos) y latente alineado por semejanza (puntos)')
        plt.show()

## Paneles para el artículo (LaTeX)

Mismo formato que el notebook 09: PNG transparentes, solo puntos, extensión común.
- `paper_figs_v3_oracle/`: réplica 1 (la misma de las Figuras 2–7); filas `oracle_ae`, `oracle_free`, `oracle_freejac` y anillos;
- `paper_figs_v3_square/`: réplica **mediana** del DMAP + AE + jac 0.1 en la cuadrícula separada; filas y anillos de los cuatro modelos y del AE iniciado en el oráculo.

Al final se comprime todo en `paper_figs_v3.zip` con las tres tablas CSV.

In [ ]:
import shutil
def exportar(args):
    r = subprocess.run([sys.executable, str(ROOT / 'export_figs.py')] + [str(a) for a in args], capture_output=True, text=True)
    print(r.stdout[-1500:]); print(r.stderr[-3000:])
fo, ff, fj = OUT['ORACLE_l0'], OUT['ORACLE_l0'], OUT['ORACLE_l0.1']
exportar(['--random', fo, '--seed', 1, '--methods', '--out', '/kaggle/working/paper_figs_v3_oracle',
          '--rows', 'oracle_ae', fo, 'oracle_ae', '--rows', 'oracle_free', ff, 'oracle_ae_free', '--rows', 'oracle_freejac', fj, 'oracle_ae_free',
          '--rings-random', 'oracle_ae', fo, 'oracle_ae', '--rings-random', 'oracle_free', ff, 'oracle_ae_free',
          '--rings-random', 'oracle_freejac', fj, 'oracle_ae_free'])
sq = OUT
exportar(['--random', sq['SQ_AE_l0.1'], '--methods', '--out', '/kaggle/working/paper_figs_v3_square',
          '--rows', 'ae', sq['SQ_AE_l0'], 'ae_warm', '--rows', 'aejac', sq['SQ_AE_l0.1'], 'ae_warm',
          '--rows', 'plainae', sq['SQ_PLAIN_AE_l0'], 'plainae', '--rows', 'oracle_free', sq['SQ_ORACLE_l0'], 'oracle_ae_free',
          '--rings-grid', 'ae', sq['SQ_AE_l0'], 'ae_warm', '--rings-grid', 'aejac', sq['SQ_AE_l0.1'], 'ae_warm',
          '--rings-grid', 'plainae', sq['SQ_PLAIN_AE_l0'], 'plainae', '--rings-grid', 'oracle_free', sq['SQ_ORACLE_l0'], 'oracle_ae_free'])
dest = Path('/kaggle/working/paper_figs_v3')
dest.mkdir(exist_ok=True)
for sub in ('paper_figs_v3_oracle', 'paper_figs_v3_square'):
    if Path('/kaggle/working', sub).exists():
        shutil.copytree(Path('/kaggle/working', sub), dest / sub, dirs_exist_ok=True)
for f in ('final_v3_tabla_normalizada.csv', 'final_v3_referencias_ambiente.csv'):
    if Path('/kaggle/working', f).exists():
        shutil.copy(Path('/kaggle/working', f), dest / f)
(dest / 'beta.json').write_text(json.dumps(dict(factor_random=FACTOR, factor_square=FACTOR_SQ,
                                                 admissible_square=sel_q.get('admissible')), indent=1))
shutil.make_archive('/kaggle/working/paper_figs_v3', 'zip', dest)
print(sorted(p.name for p in dest.rglob('*'))[:60])

## Qué descargar

**`paper_figs_v3.zip`**: tablas `final_v3_tabla_normalizada.csv` y `final_v3_referencias_ambiente.csv`, `beta.json` y los paneles para LaTeX. Con eso basta para el artículo.

Para reutilizar sin reentrenar, guarda también `/kaggle/working/results_swiss_v3_*` y `beta_sweep_swiss_square/`.